# Q1 Data Audit

Exploratory audit of the supplied Campaign Team Tracking and Coverage Reconciliation datasets. This notebook reads source files without modifying them and records descriptive findings only.

**Scope:** inventory, structure, completeness, validity, and duplicate checks only. No quality thresholds are applied, no source records are removed, and no analytical conclusions are drawn.

**Reproducibility:** input locations are resolved relative to the repository root. By default, the notebook expects the supplied data pack in its retained local location; set the `EHA_Q1_DATA_DIR` environment variable to point to another copy of the same Q1 source folder. The only generated file is the descriptive audit summary in `outputs/tables/`.

## 1. Environment and imports

Load the libraries used for tabular and spatial inspection, then resolve project-relative input and output paths. The path check stops execution with a clear message if the supplied data pack is not available.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pyogrio
import geopandas as gpd
import pandas as pd

pd.set_option('display.max_columns', 100)

def field_inventory(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a consistent descriptive inventory without altering source records."""
    return pd.DataFrame({
        'field': frame.columns,
        'data_type': frame.dtypes.astype(str).values,
        'missing_values': frame.isna().sum().values,
    })


PROJECT_NAME = 'Q1_Campaign_Team_Tracking'
cwd = Path.cwd().resolve()
try:
    REPO_ROOT = next(path for path in (cwd, *cwd.parents) if (path / PROJECT_NAME).exists())
except StopIteration as error:
    raise RuntimeError(
        f'Could not locate the repository root containing {PROJECT_NAME}. Run this notebook from within the repository.'
    ) from error
DATA_ROOT = Path(os.environ.get(
    'EHA_Q1_DATA_DIR',
    REPO_ROOT / 'Technical_asssessment' / 'eHA_Assessment_Data_Pack_v4_CANDIDATE' / 'Part1_Q1_Campaign_Tracking',
))
OUTPUT_PATH = REPO_ROOT / PROJECT_NAME / 'outputs' / 'tables' / 'data_audit_summary.csv'

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f'Source data not found at {DATA_ROOT}. Set EHA_Q1_DATA_DIR to the supplied Q1 data folder.'
    )

print(f'Repository root: {REPO_ROOT}')
print(f'Source data: {DATA_ROOT}')

Repository root: C:\Users\umary\Desktop\EHA test
Source data: C:\Users\umary\Desktop\EHA test\Technical_asssessment\eHA_Assessment_Data_Pack_v4_CANDIDATE\Part1_Q1_Campaign_Tracking


## 2. GPS track inventory

Inventory every supplied GPS CSV before any transformation. The audit records file coverage, team and date identifiers, record counts, schema variants, missing values, coordinate-range exceptions, and timestamp parse failures. Invalid or missing values are counted and retained in the source files.

In [2]:
track_files = sorted((DATA_ROOT / 'tracks').glob('*.csv'))
if not track_files:
    raise FileNotFoundError('No GPS CSV files found in the tracks directory.')

inventory_rows = []
schema_signatures = {}
for file_path in track_files:
    frame = pd.read_csv(file_path)
    parsed_timestamp = pd.to_datetime(frame['timestamp'], errors='coerce')
    signature = tuple(frame.columns)
    schema_signatures.setdefault(signature, []).append(file_path.name)
    inventory_rows.append({
        'file_name': file_path.name,
        'records': len(frame),
        'team_ids': ', '.join(sorted(frame['team_id'].dropna().astype(str).unique())),
        'dates': ', '.join(sorted(parsed_timestamp.dropna().dt.date.astype(str).unique())),
        'missing_values': int(frame.isna().sum().sum()),
        'invalid_timestamps': int(parsed_timestamp.isna().sum()),
        'longitude_out_of_range': int((~frame['longitude'].between(-180, 180)).sum()),
        'latitude_out_of_range': int((~frame['latitude'].between(-90, 90)).sum()),
    })

gps_inventory = pd.DataFrame(inventory_rows)
gps_inventory.head()

            file_name  records team_ids  \
0  T01_2026-03-09.csv    29303      T01   
1  T01_2026-03-10.csv    15129      T01   
2  T01_2026-03-11.csv    20274      T01   
3  T01_2026-03-12.csv    22305      T01   
4  T01_2026-03-13.csv    14984      T01   

                                               dates  missing_values  \
0  2026-03-09, 2026-03-10, 2026-03-11, 2026-03-12...             586   
1  2026-03-10, 2026-03-11, 2026-03-12, 2026-03-13...             302   
2  2026-03-11, 2026-03-12, 2026-03-13, 2026-03-14...             404   
3  2026-03-12, 2026-03-13, 2026-03-14, 2026-03-15...             446   
4  2026-03-13, 2026-03-14, 2026-03-15, 2026-03-16...             300   

   invalid_timestamps  longitude_out_of_range  latitude_out_of_range  
0                   0                       0                      0  
1                   0                       0                      0  
2                   0                       0                      0  
3                   0   

The next cell consolidates file-level observations into campaign-wide counts and displays the observed schemas. It does not standardize, filter, or overwrite any track records.

In [3]:
gps_totals = {
    'file_count': len(track_files),
    'team_count': gps_inventory['team_ids'].str.split(', ').explode().nunique(),
    'campaign_date_count': gps_inventory['dates'].str.split(', ').explode().nunique(),
    'total_records': int(gps_inventory['records'].sum()),
    'total_missing_values': int(gps_inventory['missing_values'].sum()),
    'invalid_timestamps': int(gps_inventory['invalid_timestamps'].sum()),
    'longitude_out_of_range': int(gps_inventory['longitude_out_of_range'].sum()),
    'latitude_out_of_range': int(gps_inventory['latitude_out_of_range'].sum()),
    'schema_variants': len(schema_signatures),
}

print('GPS track totals')
display(pd.Series(gps_totals))
print('Schema consistency')
for columns, files in schema_signatures.items():
    print(f'{len(files)} file(s): {list(columns)}')

gps_inventory[['team_ids', 'dates']].drop_duplicates().sort_values(['team_ids', 'dates'])

    team_ids                                              dates
0        T01  2026-03-09, 2026-03-10, 2026-03-11, 2026-03-12...
1        T01  2026-03-10, 2026-03-11, 2026-03-12, 2026-03-13...
2        T01  2026-03-11, 2026-03-12, 2026-03-13, 2026-03-14...
3        T01  2026-03-12, 2026-03-13, 2026-03-14, 2026-03-15...
4        T01  2026-03-13, 2026-03-14, 2026-03-15, 2026-03-16...
..       ...                                                ...
155      T32                             2026-03-09, 2026-03-10
156      T32                             2026-03-10, 2026-03-11
157      T32                             2026-03-11, 2026-03-12
158      T32                             2026-03-12, 2026-03-13
159      T32                             2026-03-13, 2026-03-14

[160 rows x 2 columns]

GPS track totals
Schema consistency
160 file(s): ['team_id', 'logger_id', 'timestamp', 'longitude', 'latitude', 'accuracy_m', 'speed_kmh']


file_count                   160
team_count                    32
campaign_date_count           24
total_records             956702
total_missing_values       19025
invalid_timestamps             0
longitude_out_of_range         0
latitude_out_of_range          0
schema_variants                1
dtype: int64

## 3. Settlement masterlist audit

Inspect the planned settlement reference data for identifier uniqueness, coordinate completeness, and administrative distribution. Counts are reported by LGA and ward to expose the structure of the supplied masterlist without changing it.

In [4]:
settlements = pd.read_csv(DATA_ROOT / 'settlement_masterlist.csv')
settlement_summary = {
    'row_count': len(settlements),
    'field_count': len(settlements.columns),
    'duplicate_settlement_ids': int(settlements['settlement_id'].duplicated().sum()),
    'missing_coordinates': int(settlements[['longitude', 'latitude']].isna().any(axis=1).sum()),
}

display(settlements.head())
display(field_inventory(settlements))
display(pd.Series(settlement_summary))
display(settlements.groupby('lga_name', dropna=False).size().rename('settlement_count').reset_index())
display(settlements.groupby(['lga_name', 'ward_name'], dropna=False).size().rename('settlement_count').reset_index())

  settlement_id settlement_name settlement_type ward_code ward_name lga_code  \
0        S00790         Uztsimi         Village      W009   Washatu    LGA02   
1        S00054         Loshaja         Village      W001   Ekngoyi    LGA01   
2        S01437          Sanami          Hamlet      W020    Famade    LGA01   
3        S01143          Loyoni      Settlement      W015    Baluru    LGA01   
4        S01213          Sadewo         Village      W015    Baluru    LGA01   

  lga_name  longitude   latitude  target_population_under5  
0   Gwarin   7.514759  11.010410                     125.0  
1  Idi-Oro   7.078284  11.081093                     173.0  
2  Idi-Oro   7.675213  11.084507                      11.0  
3  Idi-Oro   7.713549  11.284635                     131.0  
4  Idi-Oro   7.700103  11.283331                      86.0  

                      field data_type  missing_values
0             settlement_id       str               0
1           settlement_name       str               0
2           settlement_type       str               0
3                 ward_code       str               0
4                 ward_name       str               0
5                  lga_code       str               0
6                  lga_name       str               0
7                 longitude   float64               0
8                  latitude   float64               0
9  target_population_under5   float64               8

row_count                   2562
field_count                   10
duplicate_settlement_ids       0
missing_coordinates            0
dtype: int64

  lga_name  settlement_count
0   Gwarin               432
1  Idi-Oro              1046
2    Ilela               580
3  Katsuma               504

   lga_name       ward_name  settlement_count
0    Gwarin         Adwana                  1
1    Gwarin         Darari                  1
2    Gwarin   Enyoko-Kofar                  1
3    Gwarin        Sashako                  2
4    Gwarin         Uzritu                  1
..      ...             ...               ...
94  Katsuma          Satide                59
95  Katsuma          Sudefa                34
96  Katsuma          Suwade                45
97  Katsuma          TASAYI                 3
98  Katsuma          Tasayi                60

[99 rows x 3 columns]

## 4. eTally audit

Review the daily e-tally dataset for expected fields, completeness, repeated date/team/settlement combinations, and dose values requiring later investigation. The suspicious-dose check is descriptive: it flags negative, missing, or target-exceeding values without treating them as errors or deleting them.

In [5]:
etally = pd.read_csv(DATA_ROOT / 'etally_daily.csv')
etally_key = ['campaign_date', 'team_id', 'settlement_id']
etally_duplicates = etally[etally.duplicated(etally_key, keep=False)].sort_values(etally_key)
suspicious_doses = etally[
    etally['doses_administered'].isna()
    | (etally['doses_administered'] < 0)
    | (etally['doses_administered'] > etally['target_population_under5'])
].copy()
etally_summary = {
    'row_count': len(etally),
    'field_count': len(etally.columns),
    'duplicate_key_rows': len(etally_duplicates),
    'missing_values': int(etally.isna().sum().sum()),
    'suspicious_dose_rows': len(suspicious_doses),
}

display(etally.head())
display(field_inventory(etally))
display(pd.Series(etally_summary))
display(etally_duplicates.head())
display(suspicious_doses.head())

  campaign_date team_id settlement_id ward_code lga_name  \
0    2026-03-09     T01        S00033      W001  Idi-Oro   
1    2026-03-09     T01        S02144      W033  Katsuma   
2    2026-03-09     T01        S00071      W001  Idi-Oro   
3    2026-03-09     T01        S00070      W001  Idi-Oro   
4    2026-03-09     T01        S02110      W033  Katsuma   

   target_population_under5  doses_administered  
0                        31                  23  
1                        40                  23  
2                       177                 107  
3                        49                  31  
4                        32                  22  

                      field data_type  missing_values
0             campaign_date       str               0
1                   team_id       str               0
2             settlement_id       str               0
3                 ward_code       str               0
4                  lga_name       str               7
5  target_population_under5     int64               0
6        doses_administered     int64               0

row_count               2023
field_count                7
duplicate_key_rows         0
missing_values             7
suspicious_dose_rows     201
dtype: int64

Empty DataFrame
Columns: [campaign_date, team_id, settlement_id, ward_code, lga_name, target_population_under5, doses_administered]
Index: []

   campaign_date team_id settlement_id ward_code lga_name  \
13    2026-03-09     T01        S00002      W001  Idi-Oro   
30    2026-03-10     T01        S00026      W001  Idi-Oro   
33    2026-03-10     T01        S00017      W001  Idi-Oro   
55    2026-03-10     T01        S02148      W033  Katsuma   
58    2026-03-10     T01        S02155      W033  Katsuma   

    target_population_under5  doses_administered  
13                       538                 542  
30                        21                  22  
33                       392                 394  
55                        99                 101  
58                       128                 132  

## 5. Inaccessible settlements audit

Describe the supplied security-accessibility reference data, including completeness and administrative distribution. This audit does not infer causes of inaccessibility or alter the classification.

In [6]:
inaccessible = pd.read_csv(DATA_ROOT / 'inaccessible_settlements.csv')
inaccessible_summary = {
    'row_count': len(inaccessible),
    'field_count': len(inaccessible.columns),
    'missing_values': int(inaccessible.isna().sum().sum()),
}

display(inaccessible.head())
display(field_inventory(inaccessible))
display(pd.Series(inaccessible_summary))
display(inaccessible.groupby('lga_name', dropna=False).size().rename('settlement_count').reset_index())
display(inaccessible.groupby(['lga_name', 'ward_name'], dropna=False).size().rename('settlement_count').reset_index())

  settlement_id settlement_name ward_code ward_name lga_name  \
0        S00955          Datina      W013   Sashako   Gwarin   
1        S00956    Irjiba-Yamma      W013   Sashako   Gwarin   
2        S00960          Kalama      W013   Sashako   Gwarin   
3        S00961          Kajita      W013   Sashako   Gwarin   
4        S00962    Uzsana-Gabas      W013   Sashako   Gwarin   

  security_classification date_classified  
0            Inaccessible      2026-02-24  
1            Inaccessible      2026-02-24  
2    Partially accessible      2026-02-24  
3            Inaccessible      2026-02-24  
4            Inaccessible      2026-02-24  

                     field data_type  missing_values
0            settlement_id       str               0
1          settlement_name       str               0
2                ward_code       str               0
3                ward_name       str               0
4                 lga_name       str               0
5  security_classification       str               0
6          date_classified       str               0

row_count         75
field_count        7
missing_values     0
dtype: int64

  lga_name  settlement_count
0   Gwarin                15
1  Katsuma                60

  lga_name ward_name  settlement_count
0   Gwarin   Sashako                15
1  Katsuma   Longoma                 9
2  Katsuma   Sashasa                16
3  Katsuma    Satide                22
4  Katsuma    Sudefa                13

## 6. Boundary audit

Load each GeoPackage layer and report its name, feature count, coordinate reference system, missing geometries, and geometry-validity status. Geometry exceptions are reported for review; no geometries are repaired or excluded.

In [7]:
boundary_path = DATA_ROOT / 'boundaries.gpkg'
if not boundary_path.exists():
    raise FileNotFoundError(f'Boundary GeoPackage not found: {boundary_path}')

# fiona has no Python 3.14 wheel available in this environment and cannot be built
# from source without a local GDAL toolchain; pyogrio (geopandas' own default I/O
# engine) provides the same layer-listing capability and is used here instead. This
# is a tooling substitution only -- it does not change how any layer is read.
boundary_rows = []
for layer_name, _geometry_type in pyogrio.list_layers(boundary_path):
    layer = gpd.read_file(boundary_path, layer=layer_name)
    boundary_rows.append({
        'layer': layer_name,
        'feature_count': len(layer),
        'crs': str(layer.crs),
        'invalid_geometries': int((~layer.geometry.is_valid).sum()),
        'missing_geometries': int(layer.geometry.isna().sum()),
    })

boundary_audit = pd.DataFrame(boundary_rows)
display(boundary_audit)

   layer  feature_count        crs  invalid_geometries  missing_geometries
0  wards             40  EPSG:4326                   0                   0
1   lgas              4  EPSG:4326                   0                   0
2  state              1  EPSG:4326                   0                   0

## 7. Audit summary table

The table below consolidates descriptive audit findings into the required output file, `outputs/tables/data_audit_summary.csv`. It does not apply quality thresholds, modify source data, or make analytical decisions.

In [8]:
audit_summary = pd.DataFrame([
    {'dataset': 'GPS tracks', 'metric': 'File count', 'value': gps_totals['file_count']},
    {'dataset': 'GPS tracks', 'metric': 'Team count', 'value': gps_totals['team_count']},
    {'dataset': 'GPS tracks', 'metric': 'Campaign date count', 'value': gps_totals['campaign_date_count']},
    {'dataset': 'GPS tracks', 'metric': 'Total records', 'value': gps_totals['total_records']},
    {'dataset': 'GPS tracks', 'metric': 'Schema variants', 'value': gps_totals['schema_variants']},
    {'dataset': 'GPS tracks', 'metric': 'Missing values', 'value': gps_totals['total_missing_values']},
    {'dataset': 'GPS tracks', 'metric': 'Invalid timestamps', 'value': gps_totals['invalid_timestamps']},
    {'dataset': 'GPS tracks', 'metric': 'Coordinates outside valid ranges', 'value': gps_totals['longitude_out_of_range'] + gps_totals['latitude_out_of_range']},
    {'dataset': 'Settlement masterlist', 'metric': 'Row count', 'value': settlement_summary['row_count']},
    {'dataset': 'Settlement masterlist', 'metric': 'Duplicate settlement IDs', 'value': settlement_summary['duplicate_settlement_ids']},
    {'dataset': 'Settlement masterlist', 'metric': 'Records with missing coordinates', 'value': settlement_summary['missing_coordinates']},
    {'dataset': 'eTally', 'metric': 'Row count', 'value': etally_summary['row_count']},
    {'dataset': 'eTally', 'metric': 'Duplicate date/team/settlement rows', 'value': etally_summary['duplicate_key_rows']},
    {'dataset': 'eTally', 'metric': 'Missing values', 'value': etally_summary['missing_values']},
    {'dataset': 'eTally', 'metric': 'Potentially suspicious dose rows', 'value': etally_summary['suspicious_dose_rows']},
    {'dataset': 'Inaccessible settlements', 'metric': 'Row count', 'value': inaccessible_summary['row_count']},
    {'dataset': 'Inaccessible settlements', 'metric': 'Missing values', 'value': inaccessible_summary['missing_values']},
])

for row in boundary_audit.itertuples(index=False):
    audit_summary.loc[len(audit_summary)] = {
        'dataset': f'Boundary layer: {row.layer}',
        'metric': 'Feature count',
        'value': row.feature_count,
    }
    audit_summary.loc[len(audit_summary)] = {
        'dataset': f'Boundary layer: {row.layer}',
        'metric': 'Invalid geometries',
        'value': row.invalid_geometries,
    }

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
audit_summary.to_csv(OUTPUT_PATH, index=False)
display(audit_summary)
print(f'Audit summary written to: {OUTPUT_PATH}')

Audit summary written to: C:\Users\umary\Desktop\EHA test\Q1_Campaign_Team_Tracking\outputs\tables\data_audit_summary.csv


                     dataset                               metric   value
0                 GPS tracks                           File count     160
1                 GPS tracks                           Team count      32
2                 GPS tracks                  Campaign date count      24
3                 GPS tracks                        Total records  956702
4                 GPS tracks                      Schema variants       1
5                 GPS tracks                       Missing values   19025
6                 GPS tracks                   Invalid timestamps       0
7                 GPS tracks     Coordinates outside valid ranges       0
8      Settlement masterlist                            Row count    2562
9      Settlement masterlist             Duplicate settlement IDs       0
10     Settlement masterlist     Records with missing coordinates       0
11                    eTally                            Row count    2023
12                    eTally  Duplicat

## Audit Findings and Next Steps

Complete this section after running the descriptive audit. It is a record of observations and open questions, not a place to set analytical thresholds or select methods.

### Observed Data Issues

**GPS track campaign-date count.** The scenario states a five-day campaign (9-13 March 2026), and the data pack is described as one GPS file per team per day. The audit above found `campaign_date_count = 24` -- nearly five times the expected span, across a 160-file inventory that is otherwise schema-consistent with no invalid timestamps or out-of-range coordinates. This means the "one file per team per day" naming is not a hard boundary: individual files contain many days of continuous data.

This audit was not executed or reviewed when the pipeline was first built (2026-07-30); it was only run for the first time on 2026-07-31, during an unrelated read-only review, at which point the 24-date anomaly was immediately visible. Had this notebook been run and its output actually read at build time, the downstream defect below would very likely have been caught before it reached the coverage-reconciliation and cluster-analysis outputs, rather than after.

### Possible Impact

Full-file inspection (2026-07-31, recorded in `technical_decisions.md`) confirmed the impact directly: 104 of 160 files span 6-21 days at a constant one-point-per-minute rate, 85.5% of all 956,702 raw points fall outside their own file's nominal day, and for 66 of 160 team-days two different sibling files both produced GPS fixes during real duty hours on the same real campaign date -- physically impossible for one team, and silently corrupting settlement attribution wherever it occurred.

### Decisions Still Required

Recorded and resolved in `technical_decisions.md` (2026-07-31 entries): a seventh QA rule, `source_file_date_mismatch`, restricts every point to its own source file's nominal day, and the full pipeline (QA, attribution, visit classification, reconciliation, cluster analysis, cartography) was re-run against the correction.

### Follow-up Actions

None outstanding for this defect. If a future data pack is received from the same source, re-run this audit notebook and check `campaign_date_count` against the stated campaign length before proceeding to ingestion, rather than after downstream analysis is already built on it.